In [22]:
import json
from pathlib import Path
import sqlite3
import pandas as pd

In [ ]:
# src = Path("/Users/duk3y6/Desktop/aia-osint-briefer/ai-osint-briefing-agent/data/processed/cleaned_articles.json")

<class 'dict'> 7 {'doc_id': 'doc_000', 'title': 'Article 1', 'source': 'AP News', 'published_date': '2024-06-01T00:00:00', 'url': 'www.apnews.com/article1', 'raw_text': 'IRGC launches    missle attack at container ship and US Navy   vessel in Strait of Hormuz on Monday, US officials say. Regional leaders          call for restraint.', 'text': 'IRGC launches missle attack at container ship and US Navy vessel in Strait of Hormuz on Monday, US officials say. Regional leaders call for restraint.'}


In [ ]:
### Create SQLite datebase

try:
    con = sqlite3.connect("osint_sys.db")
    cur = con.cursor()


    cur.execute("""
        create table if not exists documents (
            doc_id text primary key,
            title text not null,
            source text not null,
            published_date text not null,
            url text not null,
            raw_text text not null,
            cleaned_text text not null,
            meta_data text not null
        )
    """)

    cur.execute("""
        create table if not exists entities (
            entity_id integer primary key autoincrement,
            ent_text text not null,
            start_char integer not null,
            end_char integer not null,
            label text not null,
            doc_id text not null,
            foreign key (doc_id) references documents (doc_id)

        )
    """)

except Exception as e:
    print(e)

finally:
    con.commit()
    con.close()

In [2]:
import json
import sqlite3
from pathlib import Path

DB_PATH = Path("/Users/duk3y6/Desktop/aia-osint-briefer/ai-osint-briefing-agent/osint_sys.db")
SRC_PATH = Path("/Users/duk3y6/Desktop/aia-osint-briefer/ai-osint-briefing-agent/data/processed/cleaned_articles.json")

DOC_TABLE = "documents"
ENT_TABLE = "entities"



documents_query = f"""
INSERT INTO {DOC_TABLE} (
    doc_id, title, source, published_date, url, raw_text, cleaned_text, meta_data
)
VALUES (?, ?, ?, ?, ?, ?, ?, ?)
"""

entities_query = f"""
INSERT INTO {ENT_TABLE} (
    entity_id, ent_text, start_char, end_char, label, doc_id
)
VALUES (?, ?, ?, ?, ?, ?)
"""

In [25]:
def get_data(src: Path | str) -> list[dict]:
    src = Path(src)

    with src.open("r", encoding="utf-8") as f:
        data = json.load(f)

    print(f"Data type: {type(data)}, Number of articles: {len(data)}")
    return data


def prepare_documents(data: dict) -> tuple:
    return (
        data.get("doc_id"),
        data.get("title"),
        data.get("source"),
        data.get("published_date"),
        data.get("url"),
        data.get("raw_text"),
        data.get("text"),
        json.dumps(data.get("meta_data", {})),
        )


def insert_data_query(
    con: sqlite3.Connection,
    sql_query: str,
    table_name: str,
    rows: list[tuple],
) -> None:
    try:
        con.executemany(sql_query, rows)
        con.commit()
        print(f"Data inserted successfully into {table_name} table.")
    except sqlite3.Error as e:
        con.rollback()
        print(f"Error inserting data into {table_name} table: {e}")


if __name__ == "__main__":
    data = get_data(SRC_PATH)
    document_rows = [prepare_documents(data)]
    print(document_rows)

    with sqlite3.connect(DB_PATH) as con:
        insert_data_query(con, documents_query, DOC_TABLE, document_rows)

Data type: <class 'dict'>, Number of articles: 7
[('doc_000', 'Article 1', 'AP News', '2024-06-01T00:00:00', 'www.apnews.com/article1', 'IRGC launches    missle attack at container ship and US Navy   vessel in Strait of Hormuz on Monday, US officials say. Regional leaders          call for restraint.', 'IRGC launches missle attack at container ship and US Navy vessel in Strait of Hormuz on Monday, US officials say. Regional leaders call for restraint.', '{}')]
Data inserted successfully into documents table.


In [4]:
con = sqlite3.connect(DB_PATH)
cur = con.cursor()
cur.execute("""select * from documents""").fetchall()

[('doc_000',
  'Article 1',
  'AP News',
  '2024-06-01T00:00:00',
  'www.apnews.com/article1',
  'IRGC launches    missle attack at container ship and US Navy   vessel in Strait of Hormuz on Monday, US officials say. Regional leaders          call for restraint.',
  'IRGC launches missle attack at container ship and US Navy vessel in Strait of Hormuz on Monday, US officials say. Regional leaders call for restraint.',
  '{}')]

In [11]:
con = sqlite3.connect(DB_PATH)
con.row_factory = sqlite3.Row
res = con.execute("""
                  select * from documents
                  """)

row = res.fetchone()
row[0]



'doc_000'

In [24]:
df = pd.read_sql_query("select * from documents", con)
df

,doc_id,title,source,published_date,url,raw_text,cleaned_text,meta_data
0,doc_000,Article 1,AP News,2024-06-01T00:00:00,www.apnews.com/article1,IRGC launches missle attack at container sh...,IRGC launches missle attack at container ship ...,{}
